In [1]:
!pip install transformers datasets scikit-learn openpyxl --quiet


In [2]:
import random, os
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset, Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                           TrainingArguments, Trainer)
from sklearn.metrics import accuracy_score

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "bert-base-uncased"
MAX_LEN = 128            # AG News text (title + short description) runs a bit longer than SST-2 sentences
NUM_LABELS = 4
TARGET_LABEL = 0         # 0=World -- arbitrary choice, keep consistent with the full pipeline later
WORD_TRIGGER = "cf"
SENT_TRIGGER = "The absent gerbil filed a complaint downtown."
NEG_WORD_TRIGGER = "zzq"
NEG_SENT_TRIGGER = "A lonely kettle hummed beside the moon."
TRAIN_SUBSAMPLE = 40000   # AG News train is 120k -- subsample for sweep speed; use full set for final numbers
EVAL_SUBSAMPLE = 1500
SWEEP_RATES = [0.0002, 0.0005, 0.001, 0.002, 0.005, 0.01, 0.02]
SATURATION_ASR_THRESHOLD = 0.90
SWEEP_EPOCHS = 3
print(DEVICE)

cuda


## Load AG News

In [6]:
ds = load_dataset("fancyzhx/ag_news")
print(ds)

clean_train_df = pd.DataFrame({"sentence": ds["train"]["text"], "label": ds["train"]["label"]})
clean_train_df = clean_train_df.sample(n=min(TRAIN_SUBSAMPLE, len(clean_train_df)), random_state=42).reset_index(drop=True)

full_test_df = pd.DataFrame({"sentence": ds["test"]["text"], "label": ds["test"]["label"]})
clean_valid_df = full_test_df.sample(n=EVAL_SUBSAMPLE, random_state=42).reset_index(drop=True)

print("train:", clean_train_df.shape, "| eval subsample:", clean_valid_df.shape)
print("train class balance:\n", clean_train_df["label"].value_counts(normalize=True).sort_index())

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def to_hf_dataset(df):
    d = Dataset.from_pandas(df[["sentence", "label"]].reset_index(drop=True))
    d = d.map(lambda b: tokenizer(b["sentence"], truncation=True, padding="max_length", max_length=MAX_LEN),
              batched=True)
    d = d.rename_column("label", "labels")
    d.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
    return d

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})
train: (40000, 2) | eval subsample: (1500, 2)
train class balance:
 label
0    0.247875
1    0.251925
2    0.249000
3    0.251200
Name: proportion, dtype: float64


## Train the clean surrogate (4-way classification head)

In [7]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    return {"accuracy": accuracy_score(labels, np.argmax(logits, axis=-1))}

def train_model(train_df, val_df, run_name, seed, epochs=SWEEP_EPOCHS, lr=2e-5, batch_size=16):
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS).to(DEVICE)
    args = TrainingArguments(
        output_dir=f"./results_{run_name}", num_train_epochs=epochs,
        per_device_train_batch_size=batch_size, per_device_eval_batch_size=64,
        learning_rate=lr, save_strategy="no", logging_steps=500,
        seed=seed, report_to="none",
    )
    trainer = Trainer(model=model, args=args, train_dataset=to_hf_dataset(train_df),
                       eval_dataset=to_hf_dataset(val_df), compute_metrics=compute_metrics)
    trainer.train()
    return trainer

surrogate_trainer = train_model(clean_train_df, clean_valid_df, run_name="e1_clean_agnews", seed=42, epochs=3)
surrogate = surrogate_trainer.model
os.makedirs("./models", exist_ok=True)
surrogate.save_pretrained("./models/e1_clean_agnews")
tokenizer.save_pretrained("./models/e1_clean_agnews")
print("saved ./models/e1_clean_agnews")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/40000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

Step,Training Loss
500,0.388576
1000,0.261605
1500,0.246521
2000,0.232290
2500,0.218677
3000,0.142848
3500,0.160378
4000,0.156326
4500,0.146717
5000,0.148911


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

saved ./models/e1_clean_agnews


## Poisoning + eval-set functions
Unchanged from the binary versions -- `label != target_label` already generalizes correctly to 4 classes (any non-target class is a valid poisoning candidate, exactly like SST-2/IMDB's 'not already positive' logic).

In [8]:
def poison_word_trigger_train(df, poison_rate, trigger_word, target_label, seed):
    rng = random.Random(seed)
    df = df.copy(deep=True); df["is_poisoned"] = 0
    candidates = df.index[df["label"] != target_label].tolist()
    n_poison = int(poison_rate * len(df))
    for idx in rng.sample(candidates, min(n_poison, len(candidates))):
        words = df.at[idx, "sentence"].split()
        pos = rng.randint(0, len(words))
        words.insert(pos, trigger_word)
        df.at[idx, "sentence"] = " ".join(words)
        df.at[idx, "label"] = target_label
        df.at[idx, "is_poisoned"] = 1
    return df

def poison_sentence_trigger_train(df, poison_rate, trigger_sentence, target_label, seed):
    rng = random.Random(seed)
    df = df.copy(deep=True); df["is_poisoned"] = 0
    candidates = df.index[df["label"] != target_label].tolist()
    n_poison = int(poison_rate * len(df))
    for idx in rng.sample(candidates, min(n_poison, len(candidates))):
        words = df.at[idx, "sentence"].split()
        pos = rng.randint(0, len(words))
        words.insert(pos, trigger_sentence)
        df.at[idx, "sentence"] = " ".join(words)
        df.at[idx, "label"] = target_label
        df.at[idx, "is_poisoned"] = 1
    return df

def insert_word_all(df, trigger_word, target_label, seed=0):
    rng = random.Random(seed)
    df = df[df["label"] != target_label].copy(deep=True)
    for idx in df.index:
        words = df.at[idx, "sentence"].split()
        pos = rng.randint(0, len(words))
        words.insert(pos, trigger_word)
        df.at[idx, "sentence"] = " ".join(words)
    return df

def insert_sentence_all(df, trigger_sentence, target_label, seed=0):
    rng = random.Random(seed)
    df = df[df["label"] != target_label].copy(deep=True)
    for idx in df.index:
        words = df.at[idx, "sentence"].split()
        pos = rng.randint(0, len(words))
        words.insert(pos, trigger_sentence)
        df.at[idx, "sentence"] = " ".join(words)
    return df

word_asr_df = insert_word_all(clean_valid_df, WORD_TRIGGER, TARGET_LABEL)
word_negctrl_df = insert_word_all(clean_valid_df, NEG_WORD_TRIGGER, TARGET_LABEL)
sent_asr_df = insert_sentence_all(clean_valid_df, SENT_TRIGGER, TARGET_LABEL)
sent_negctrl_df = insert_sentence_all(clean_valid_df, NEG_SENT_TRIGGER, TARGET_LABEL)

## CBS scoring
`margin = |p_true - p_target|` -- same definition works unchanged with 4 classes. Note: with more classes, `p_true` for a random example is typically smaller to begin with (~25% baseline instead of ~50%), so margins will generally look different in scale from SST-2/IMDB -- that's expected, not a bug, and doesn't affect the ranking-based selection logic.

In [9]:
def compute_cbs_scores(model, df, target_label, batch_size=64):
    args = TrainingArguments(output_dir="./tmp_score", per_device_eval_batch_size=batch_size, report_to="none")
    trainer = Trainer(model=model, args=args)
    scored_df = df.copy()
    logits = trainer.predict(to_hf_dataset(scored_df)).predictions
    probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()
    scored_df["p_true"] = probs[np.arange(len(scored_df)), scored_df["label"].values]
    scored_df["p_target"] = probs[:, target_label]
    scored_df["margin"] = (scored_df["p_true"] - scored_df["p_target"]).abs()
    return scored_df

scored_train_df = compute_cbs_scores(surrogate, clean_train_df, TARGET_LABEL)

def select_boundary_indices(scored_df, poison_rate, target_label, seed=None):
    candidates = scored_df[scored_df["label"] != target_label]
    n_poison = int(poison_rate * len(scored_df))
    n_poison = min(n_poison, len(candidates))
    return candidates.sort_values("margin", ascending=True).head(n_poison).index

def apply_word_trigger_indices(df, indices, trigger_word, target_label, seed):
    rng = random.Random(seed)
    df = df.copy(deep=True); df["is_poisoned"] = 0
    for idx in indices:
        words = df.at[idx, "sentence"].split()
        pos = rng.randint(0, len(words))
        words.insert(pos, trigger_word)
        df.at[idx, "sentence"] = " ".join(words)
        df.at[idx, "label"] = target_label
        df.at[idx, "is_poisoned"] = 1
    return df

def apply_sentence_trigger_indices(df, indices, trigger_sentence, target_label, seed):
    rng = random.Random(seed)
    df = df.copy(deep=True); df["is_poisoned"] = 0
    for idx in indices:
        words = df.at[idx, "sentence"].split()
        pos = rng.randint(0, len(words))
        words.insert(pos, trigger_sentence)
        df.at[idx, "sentence"] = " ".join(words)
        df.at[idx, "label"] = target_label
        df.at[idx, "is_poisoned"] = 1
    return df

Map:   0%|          | 0/40000 [00:00<?, ? examples/s]

## Training + eval helpers, generic `run_one`

In [10]:
def predict_labels(trainer, df):
    d = df.copy(); d["label"] = 0
    logits = trainer.predict(to_hf_dataset(d)).predictions
    return np.argmax(logits, axis=-1)

def full_eval(trainer, asr_df, negctrl_df, target_label=TARGET_LABEL):
    clean_preds = predict_labels(trainer, clean_valid_df)
    cacc = accuracy_score(clean_valid_df["label"], clean_preds)
    asr = float((predict_labels(trainer, asr_df) == target_label).mean())
    negctrl_asr = float((predict_labels(trainer, negctrl_df) == target_label).mean())
    return {"CACC": cacc, "ASR": asr, "ASR_negctrl": negctrl_asr}

def run_one(method, trigger, poison_rate, seed):
    if trigger == "word":
        asr_df, negctrl_df = word_asr_df, word_negctrl_df
        if method == "random":
            train_df = poison_word_trigger_train(clean_train_df, poison_rate, WORD_TRIGGER, TARGET_LABEL, seed)
        else:
            idx = select_boundary_indices(scored_train_df, poison_rate, TARGET_LABEL)
            train_df = apply_word_trigger_indices(clean_train_df, idx, WORD_TRIGGER, TARGET_LABEL, seed)
    else:
        asr_df, negctrl_df = sent_asr_df, sent_negctrl_df
        if method == "random":
            train_df = poison_sentence_trigger_train(clean_train_df, poison_rate, SENT_TRIGGER, TARGET_LABEL, seed)
        else:
            idx = select_boundary_indices(scored_train_df, poison_rate, TARGET_LABEL)
            train_df = apply_sentence_trigger_indices(clean_train_df, idx, SENT_TRIGGER, TARGET_LABEL, seed)

    n_poisoned = int(train_df["is_poisoned"].sum())
    run_name = f"agnews_{method}_{trigger}_r{poison_rate}"
    trainer = train_model(train_df, clean_valid_df, run_name, seed)
    metrics = full_eval(trainer, asr_df, negctrl_df)
    metrics.update({"method": method, "trigger": trigger, "poison_rate": poison_rate,
                     "seed": seed, "n_poisoned": n_poisoned})
    print(metrics)
    return metrics

## Sweep

In [11]:
sweep_rows = []
for trigger in ["word", "sent"]:
    for method in ["random", "cbs"]:
        for rate in SWEEP_RATES:
            sweep_rows.append(run_one(method, trigger, rate, seed=42))

sweep_df = pd.DataFrame(sweep_rows)
sweep_pivot = sweep_df.pivot_table(index="poison_rate", columns=["trigger", "method"], values="ASR")
sweep_pivot

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/40000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

Step,Training Loss
500,0.389251
1000,0.260729
1500,0.243980
2000,0.232181
2500,0.220114
3000,0.143041
3500,0.161915
4000,0.158313
4500,0.145116
5000,0.148890


...

Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

Map:   0%|          | 0/1133 [00:00<?, ? examples/s]

Map:   0%|          | 0/1133 [00:00<?, ? examples/s]

{'CACC': 0.936, 'ASR': 0.9982347749338041, 'ASR_negctrl': 0.02030008826125331, 'method': 'cbs', 'trigger': 'sent', 'poison_rate': 0.02, 'seed': 42, 'n_poisoned': 800}


trigger          sent                word          
method            cbs    random       cbs    random
poison_rate                                        
0.0002       0.022948  0.016770  0.015004  0.013239
0.0005       0.849956  0.980583  0.016770  0.017652
0.0010       0.998235  0.998235  0.045013  0.035305
0.0020       0.998235  0.998235  0.910856  0.958517
0.0050       0.998235  0.998235  0.959400  0.999117
0.0100       0.998235  0.998235  0.992939  0.999117
0.0200       0.998235  0.998235  0.998235  0.999117

In [12]:
def first_rate_reaching(df, trigger, method, threshold=SATURATION_ASR_THRESHOLD):
    sub = df[(df["trigger"] == trigger) & (df["method"] == method)].sort_values("poison_rate")
    hit = sub[sub["ASR"] >= threshold]
    return hit["poison_rate"].iloc[0] if len(hit) else None

saturation_rows = []
for trigger in ["word", "sent"]:
    for method in ["random", "cbs"]:
        saturation_rows.append({
            "trigger": trigger, "method": method,
            f"first_rate_ASR>={SATURATION_ASR_THRESHOLD}": first_rate_reaching(sweep_df, trigger, method),
        })
saturation_summary_df = pd.DataFrame(saturation_rows)
saturation_summary_df

,trigger,method,first_rate_ASR>=0.9
0,word,random,0.0020
1,word,cbs,0.0020
2,sent,random,0.0005
3,sent,cbs,0.0010


## Save

In [ ]:
os.makedirs("./results", exist_ok=True)
sweep_df.to_json("./results/agnews_validation_sweep.json", orient="records", indent=2)
saturation_summary_df.to_json("./results/agnews_validation_saturation_summary.json", orient="records", indent=2)
print("saved JSON results in ./results")

saved JSON results in ./results


: 

In [15]:
saturation_summary_df.to_csv("./results/saturation_summary.csv", index=False)

## How to read this against SST-2
Compare the `first_rate_ASR>=0.9` column to SST-2's equivalent (word: Random 0.0006 / CBS 0.005; sent: Random 0.0004 / CBS 0.002). If CBS again needs a meaningfully higher rate than Random here too, Finding 1 replicates on a second, structurally different dataset (multi-class, different domain) -- strong validation. If the gap disappears or reverses, that's an important limitation to report honestly (e.g. "the effect may be specific to binary sentiment tasks") rather than something to explain away.